[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb)

**Run order (do these once, then repeat step 3 after every work session)**

| Step | What | Notebook |
|---|---|---|
| 0 | Create a GitHub fine-grained token for this repo only (Contents + Workflows read/write, Actions read). In Colab: key icon (Secrets) > add `GITHUB_TOKEN` > turn on Notebook access. Never paste it in a cell or in chat. | none |
| 1 | Set up CI/CD once: writes `.github/workflows`, dry-runs lint and tests, pushes, reads run status. | [01 CICD](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_01_CICD_Pipeline.ipynb) |
| 2 | First sync as a dry run (`DRY_RUN = True` is now the default): check the staged file list. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb) |
| 3 | Real sync: `DRY_RUN = False`. Run top to bottom; the last cell pushes. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb) |

Within each notebook, run cells top to bottom (Runtime > Run all is fine). The last cell is always the push.

# Rodic_00_GitHub_Sync
Pushes everything your working Colab produced to `Subhash-Pal/Rodic-infra-Colab`, into one `Vx/` folder.

**How to use**
1. Add Colab Secret `GITHUB_TOKEN` (key icon, left sidebar, enable Notebook access). Fine-grained PAT: Contents read/write on this repo only. Add Workflows read/write if you also push `.github/workflows/` (used by notebook 01).
2. Colab notebooks do not share files (each has its own runtime), so Google Drive is the bridge. In your working Colab, mount Drive and write code to `/content/drive/MyDrive/rodic_work/`. To push whole notebooks, list their file names in `NOTEBOOK_FILES` (they are read from `My Drive/Colab Notebooks`).
3. Give each working cell a markdown heading starting with `### ` that says what it adds. Headings from the notebooks in `NOTEBOOK_FILES` become the commit message and `CHANGELOG.md`.
4. Run this notebook top to bottom. The last cell is the push.

Safety: secret scan, 50 MB file guard, optional pytest gate, dry-run switch. Only `VERSION_DIR/` and `notebooks/` are staged.

**Stale-tab warning:** if you opened this notebook before 18 Nov 07:45 IST, close the tab and reopen it from the badge. The first config cell prints `Running: 00-sync v0.2`; if you do not see that, you are running an old copy. Do not use File > Save a copy in GitHub from an old tab: it overwrites the newer version.

### Config and authentication helpers
Defines repo, target folder, mounts Google Drive (the shared space between notebooks), and a `git()` wrapper that sends the token as a header, never stores it in the remote URL, and masks it in output.

In [ ]:
NB_VERSION = "00-sync v0.2 (Drive-based work folder)"
print("Running:", NB_VERSION)
import os, re, json, base64, shutil, subprocess, pathlib, datetime
from google.colab import userdata

# ---------- CONFIG: edit only this block ----------
REPO         = "Subhash-Pal/Rodic-infra-Colab"
BRANCH       = "main"
VERSION_DIR  = "v4"                 # folder inside the repo that receives your work (Vx/ rule)
WORK_DIR     = "/content/drive/MyDrive/rodic_work"   # Drive folder with .py/.yaml/.md files to push
NOTEBOOK_DIR = "/content/drive/MyDrive/Colab Notebooks"  # where Colab saves your notebooks in Drive
NOTEBOOK_FILES = []                 # notebooks to push into notebooks/, e.g. ["Rodic_Edge_CV.ipynb"]
RUN_TESTS    = True                 # run pytest before pushing, if tests exist
DRY_RUN      = True                 # True = stage + show what would be committed; set False to push
MAX_FILE_MB  = 50
# --------------------------------------------------

# Each Colab notebook runs in its OWN runtime, so /content is not shared between notebooks.
# Google Drive is the shared space: your working notebooks save files there, this one pushes them.
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/_repo"
TOKEN = userdata.get("GITHUB_TOKEN")
_B64 = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
AUTH = ["-c", f"http.extraheader=AUTHORIZATION: basic {_B64}"]

def _mask(t): return t.replace(TOKEN, "***").replace(_B64, "***")

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"] + (AUTH if auth else []) + list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = _mask(r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

### Clone or update the repo
Fresh clone on first run, otherwise fetch and rebase so your push does not collide with remote changes.

In [ ]:
if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR, cwd="/content", auth=True)
else:
    git("fetch", "origin", BRANCH, auth=True)
    git("checkout", BRANCH)
    git("pull", "--rebase", "origin", BRANCH, auth=True)
git("config", "user.name", "Subhash Pal (Colab)")
git("config", "user.email", "colab-sync@users.noreply.github.com")
print(git("log", "--oneline", "-5"))

### Secret scanner
Blocks the push if any file contains a Gemini/Google key, GitHub token, Anthropic key or private key block. Relevant because your six `GEMINI_API_KEY*` values must never reach a public repo.

In [ ]:
SECRET_PATTERNS = {
    "Google/Gemini API key": r"AIza[0-9A-Za-z_\-]{35}",
    "GitHub token":          r"(ghp_|gho_|ghs_)[A-Za-z0-9]{36}|github_pat_[A-Za-z0-9_]{50,}",
    "Anthropic key":         r"sk-ant-[A-Za-z0-9_\-]{20,}",
    "Private key block":     r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
}
TEXT_EXT = {".py",".md",".txt",".yaml",".yml",".json",".ipynb",".toml",".ini",".cfg",".env",".sh",".csv"}

def scan_for_secrets(root):
    hits = []
    for p in pathlib.Path(root).rglob("*"):
        if p.is_file() and (p.suffix.lower() in TEXT_EXT or p.name.startswith(".env")):
            try: txt = p.read_text(errors="ignore")
            except Exception: continue
            for label, pat in SECRET_PATTERNS.items():
                if re.search(pat, txt):
                    hits.append((str(p), label))
    return hits
print("Secret scanner ready.")

### Stage working files into the version folder
Copies the Drive `WORK_DIR` into `VERSION_DIR/`, ignoring caches, `.env` and key files, dropping files over the size limit, and optionally copying the working notebook into `notebooks/`.

In [ ]:
IGNORE = shutil.ignore_patterns("__pycache__", "*.pyc", ".ipynb_checkpoints", ".env", ".env.*",
                                "*.key", "*.pem", "sample_data", ".git", "venv", ".venv")
dest = pathlib.Path(REPO_DIR) / VERSION_DIR
src = pathlib.Path(WORK_DIR)
src.mkdir(parents=True, exist_ok=True)          # created on first run so this cell never fails
nb_paths = [pathlib.Path(NOTEBOOK_DIR) / n for n in NOTEBOOK_FILES]
missing = [str(p) for p in nb_paths if not p.exists()]
nb_paths = [p for p in nb_paths if p.exists()]

work_files = [p for p in src.rglob("*") if p.is_file()]
if not work_files and not nb_paths:
    print(f"Nothing to stage yet.\n"
          f"  - Put files to push into: {WORK_DIR}\n"
          f"  - and/or list notebooks in NOTEBOOK_FILES (they are read from {NOTEBOOK_DIR}).")
if missing:
    print("Notebook not found in Drive:", *missing, sep="\n  ")

skipped_big = [str(p) for p in work_files if p.stat().st_size > MAX_FILE_MB * 1024 * 1024]
if work_files:
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dest, ignore=IGNORE, dirs_exist_ok=True)
    for b in skipped_big:                       # never push huge weights/datasets by accident
        (dest / pathlib.Path(b).relative_to(src)).unlink(missing_ok=True)
    print(f"Staged {len(work_files) - len(skipped_big)} file(s) from {WORK_DIR} -> {VERSION_DIR}/")

if nb_paths:
    nbdir = pathlib.Path(REPO_DIR) / "notebooks"
    nbdir.mkdir(exist_ok=True)
    for p in nb_paths:
        shutil.copy(p, nbdir / p.name)
    print("Staged notebooks ->", "notebooks/", [p.name for p in nb_paths])
if skipped_big:
    print(f"Skipped (> {MAX_FILE_MB} MB, use Releases/Drive/HF instead):", *skipped_big, sep="\n  ")


### Build changelog and commit message from cell headings
Reads every `### ` markdown heading in the notebooks listed in `NOTEBOOK_FILES` and records it as a dated CHANGELOG entry.

In [ ]:
def headings_from_notebook(path):
    """Collect '### ...' markdown headings = what each cell added."""
    if not path or not os.path.exists(path): return []
    nbj = json.load(open(path))
    out = []
    for c in nbj.get("cells", []):
        if c["cell_type"] == "markdown":
            first = "".join(c["source"]).strip().splitlines()[:1]
            if first and first[0].startswith("###"):
                out.append(first[0].lstrip("# ").strip())
    return out

added = []
for p in nb_paths:
    added += [f"{p.stem}: {h}" for h in headings_from_notebook(p)]
stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=5, minutes=30))).strftime("%Y-%m-%d %H:%M IST")
if added:
    dest.mkdir(parents=True, exist_ok=True)
    cl = dest / "CHANGELOG.md"
    old = cl.read_text() if cl.exists() else f"# Changelog ({VERSION_DIR})\n"
    entry = f"\n## {stamp}\n" + "\n".join(f"- {h}" for h in added) + "\n"
    cl.write_text(old + entry)
    print(entry)
else:
    print("No '### ' headings staged (list notebooks in NOTEBOOK_FILES to build the changelog).")
COMMIT_MSG = f"{VERSION_DIR}: " + ("; ".join(added)[:180] if added else f"colab sync {stamp}")


### Test gate
Runs pytest on the staged folder if tests exist. A failing test blocks the push.

In [ ]:
tests_dir = pathlib.Path(REPO_DIR) / VERSION_DIR
has_tests = any(tests_dir.rglob("test_*.py")) or any(tests_dir.rglob("*_test.py"))
if RUN_TESTS and has_tests:
    subprocess.run("pip -q install pytest pyyaml scikit-image scipy", shell=True)
    r = subprocess.run(["python", "-m", "pytest", "-q", str(tests_dir)], cwd=REPO_DIR, capture_output=True, text=True)
    print(r.stdout[-2500:])
    TESTS_OK = (r.returncode == 0)
else:
    print("Tests skipped (disabled or none found).")
    TESTS_OK = True
print("TESTS_OK =", TESTS_OK)

### FINAL CELL: commit and push
Runs the secret scan and test gate, then commits, rebases and pushes. Set `DRY_RUN = True` in the config cell to preview only.

In [ ]:
hits = scan_for_secrets(REPO_DIR + "/" + VERSION_DIR) + (scan_for_secrets(REPO_DIR + "/notebooks") if os.path.isdir(REPO_DIR + "/notebooks") else [])
if hits:
    print("BLOCKED: possible secrets found. Remove them, rotate the key, then re-run.")
    for f, l in hits: print("  ", l, "->", f)
elif not TESTS_OK:
    print("BLOCKED: tests failed. Fix them or set RUN_TESTS=False knowingly.")
else:
    if os.path.isdir(REPO_DIR + "/" + VERSION_DIR): git("add", "-A", VERSION_DIR)
    if os.path.isdir(REPO_DIR + "/notebooks"): git("add", "-A", "notebooks")
    status = git("status", "--short")
    if not status:
        print("Nothing changed - nothing to push.")
    elif DRY_RUN:
        print("DRY RUN - would commit:\n" + status)
    else:
        git("commit", "-m", COMMIT_MSG)
        git("pull", "--rebase", "origin", BRANCH, auth=True)
        git("push", "origin", BRANCH, auth=True)
        print("Pushed:", git("log", "--oneline", "-1"))
        print(f"https://github.com/{REPO}/tree/{BRANCH}/{VERSION_DIR}")